# Dimer Case2:  Functions

This page contains parameter values and functions used in other pages of the chapter.

In [1]:
import matplotlib.pyplot as plt
import numpy as np
from qutip import *

**Constant parameters**

In [2]:
# default energy scale
omega0 = 1.0

# default spontaneous emission rate
gamma0 = 0.1

def scale(omega0):
    k0 = omega0  # omega = c k, c=1
    lambda0 = 2*np.pi/k0
    return k0, lambda0


**Spin operators in 2^3 = 8 dimensions**

In [3]:
# Spin operators ---
def spin_ops():
    i2 = qeye(2)
    sz = [tensor(sigmaz(),i2),tensor(i2,sigmaz())]
    sp = [tensor(sigmap(),i2),tensor(i2,sigmap())]
    sm = [tensor(sigmam(),i2),tensor(i2,sigmam())]
    return sz, sp, sm

**Product basis**

In [4]:
def pbasis():
    #--This function calls kprod3--
    # v = product basis vectors [8][8]
    # label = names of corresponding states [8]
    label=["ee","eg","ge","gg"]
    v=[]
    v.append(tensor(basis(2,0),basis(2,0)))
    v.append(tensor(basis(2,0),basis(2,1)))
    v.append(tensor(basis(2,1),basis(2,0)))
    v.append(tensor(basis(2,1),basis(2,1)))

    return v, label

**Dicke basis**

In [5]:
def dicke(v):
    #--- v = product basis [4][4]
    #--- u = Dicke basis [4][4]
    label=["e","s","a","g"]
    u=[]
    u.append(v[0])
    u.append(1/np.sqrt(2)*(v[1]+v[2]))
    u.append(1/np.sqrt(2)*(v[1]-v[2]))
    u.append(v[3]) 
    return u, label

**Dipole-dipole coupling**


In [6]:
def ddcoupling(x,chi):
    y=(1-np.cos(chi)**2)*np.cos(x)/x + (1-3*np.sin(chi)**2)*(np.sin(x)/x**2 - np.cos(x)/x**3)
    return -y*3/4

**Hamiltonian**

In [7]:
def hamiltonian():
    return 0*tensor(qeye(2),qeye(2))

In [8]:
def nbar_all(w,T):
    nbar = []
    q = w[0]
    for k in range(2):
        nbar.append(1/(np.exp(q[k]/T)-1))
    q = w[1]
    for k in range(2):
        nbar.append(1/(np.exp(q[k]/T)-1))
    return nbar

**Decay rates**

In [9]:
def decay_rate(x,chi,omega0,w):
    y=(1-np.cos(chi)**2)*np.sin(x)/x + (1-3*np.cos(chi)**2)*(np.cos(x)/x**2 - np.sin(x)/x**3)
    gamma1 = (w[0]/omega0)**3
    gamma2 = (w[1]/omega0)**3
    gamma = [(1+y)*gamma1[0],(1+y)*gamma1[1], (1-y)*gamma2[0], (1-y)*gamma2[1]]
    return gamma

**Jump and Collaps operators**

In [10]:
def collapse_ops(u,gamma,nbar):
    
    [e,s,a,g]=[0,1,2,3]

    L = []
    # superradient channel
    L.append(u[s]*u[e].dag())
    L.append(u[g]*u[s].dag())
    # subradient channel
    L.append(u[a]*u[e].dag())
    L.append(u[g]*u[a].dag())
    
    c_ops = []
    # emission operators
    for k in range(4):
        c_ops.append(np.sqrt(gamma[k]*(nbar[k]+1))*L[k])
    # absorption operators
    for k in range(4):
        c_ops.append(np.sqrt(gamma[k]*nbar[k])*L[k].dag())

    return c_ops # collapse operators including the decay rates [8]

**Energy eigenvalues**

In [11]:
def eigen_energies(omega0,Omega):
    E=[omega0, Omega, -Omega, -omega0]
    order = np.argsort(E, descending=True)
    return np.array(E), order

**Transition energy**

In [12]:
def transition_energies(E):
    # w = transition energies for each channel [3]
    [e,s,a,g]=[0,1,2,3]
    w0 = np.array([E[e]-E[s], E[s]-E[g]])
    w1 = np.array([E[e]-E[a], E[a]-E[g]])
    w = [w0,w1]
    return w

**position of emitters**

In [13]:
def emitters_pos(r12):
    # returns the position of emitters as numpy array [3][3]
    # coordinate origin = center of the triangle
    r1 = np.array([r12/2, 0, 0])
    r2 = np.array([-r12/2,0,0])
    return np.array([r1,r2])

**location of detector**

In [14]:
def detector_pos(theta,phi):
    n=np.array([np.sin(theta)*np.cos(phi),np.sin(theta)*np.sin(phi),np.cos(theta)])
    return n

In [15]:
def get_Gibbsstate(E,T):
    Boltzmann = [np.exp(-E[k]/T) for k in range(4)]
    Z = sum(Boltzmann)
    Gibbs = Boltzmann/Z
    return Gibbs